# Actividad 1 — Map-Reduce en Python puro

En esta actividad aprenderás a procesar datos usando el paradigma **Map-Reduce** con las herramientas nativas de Python:
- `map()` — transforma cada elemento de una colección (fase **Map**)
- `functools.reduce()` — combina todos los elementos en un único resultado (fase **Reduce**)

**Restricción:** está prohibido usar `pandas`, `pyspark` u otras librerías de procesamiento de datos.

---

## ¿Qué es Map-Reduce?

El paradigma Map-Reduce divide el procesamiento de datos en dos fases:

```
Datos  →  MAP (transformar / etiquetar)  →  pares (clave, valor)
                                              ↓
                                         REDUCE (agregar / combinar)
                                              ↓
                                          Resultado
```

| Función | Fase | Rol |
|---------|------|-----|
| `map(f, it)` | **Map** | Aplica `f` a cada elemento y produce pares `(clave, valor)` |
| `reduce(f, it)` | **Reduce** | Combina acumulativamente todos los pares por clave |

---

## Datos disponibles

| Archivo | Columnas |
|---------|----------|
| `usuarios.csv` | `id, nombre, direccion, tarjeta_credito` |
| `frutas.csv` | `id, nombre, precio_unitario` |
| `compras.csv` | `id_compra, id_usuario, id_producto, fecha_compra, cantidad` |

## 0. Carga de datos

Usamos el módulo `csv` de la biblioteca estándar para leer los archivos. Cada fila queda como un diccionario.

In [7]:
import csv
from functools import reduce

def leer_csv(ruta):
    """Lee un archivo CSV y retorna una lista de diccionarios."""
    with open(ruta, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

usuarios = leer_csv('usuarios.csv')
frutas   = leer_csv('frutas.csv')
compras  = leer_csv('compras.csv')

print(f"Usuarios : {len(usuarios)} registros")
print(f"Frutas   : {len(frutas)} registros")
print(f"Compras  : {len(compras)} registros")

print("\nEjemplo de compra :", compras[0])
print("Ejemplo de fruta  :", frutas[0])
print("Ejemplo de usuario:", usuarios[0])

Usuarios : 10 registros
Frutas   : 15 registros
Compras  : 361 registros

Ejemplo de compra : {'id_compra': '1', 'id_usuario': '3', 'id_producto': '15', 'fecha_compra': '2023-02-01', 'cantidad': '1'}
Ejemplo de fruta  : {'id': '1', 'nombre': 'Manzana', 'precio_unitario': '10'}
Ejemplo de usuario: {'id': '1', 'nombre': 'Juan Perez', 'direccion': 'Calle Falsa 123', 'tarjeta_credito': '1234-5678-9012-3456'}


---

## Ejercicio 1 — Cantidad total de unidades vendidas

**Objetivo:** calcular cuántas unidades en total se vendieron en todas las compras.

**Pistas:**
- Usa `map` para extraer el campo `cantidad` de cada compra (conviértelo a `int`).
- Usa `reduce` para sumar todos esos valores.

**Resultado esperado:** un único número entero.

In [8]:
# Tu solución aquí
cantidades = map(lambda c: int(c['cantidad']), compras)
total_unidades = reduce(lambda acc, x: acc + x, cantidades)
print("Total de unidades vendidas:", total_unidades)

Total de unidades vendidas: 2015


<details>
<summary>💡 Ver solución</summary>

```python
cantidades = map(lambda c: int(c['cantidad']), compras)
total_unidades = reduce(lambda acc, x: acc + x, cantidades)
print("Total de unidades vendidas:", total_unidades)
```

</details>

-----
## Ejercicio 2 — Cantidad total comprada por usuario

In [9]:
pares = map(lambda c: (c['id_usuario'], int(c['cantidad'])), compras)

def acumular(acc, par):
    """Reducer genérico: suma par[1] al bucket acc[par[0]]."""
    clave, valor = par
    acc[clave] = acc.get(clave, 0) + valor
    return acc

unidades_por_usuario = reduce(acumular, pares, {})

for uid, total in sorted(unidades_por_usuario.items(), key=lambda x: int(x[0])):
    print(f"  Usuario {uid}: {total} unidades")

  Usuario 1: 182 unidades
  Usuario 2: 217 unidades
  Usuario 3: 160 unidades
  Usuario 4: 208 unidades
  Usuario 5: 261 unidades
  Usuario 6: 169 unidades
  Usuario 7: 234 unidades
  Usuario 8: 234 unidades
  Usuario 9: 207 unidades
  Usuario 10: 143 unidades


<details>
<summary>💡 Ver solución</summary>

```python
pares = map(lambda c: (c['id_usuario'], int(c['cantidad'])), compras)

def acumular(acc, par):
    """Reducer genérico: suma par[1] al bucket acc[par[0]]."""
    clave, valor = par
    acc[clave] = acc.get(clave, 0) + valor
    return acc

unidades_por_usuario = reduce(acumular, pares, {})

for uid, total in sorted(unidades_por_usuario.items(), key=lambda x: int(x[0])):
    print(f"  Usuario {uid}: {total} unidades")
```

</details>

---

## Ejercicio 3 — Gasto total por usuario

**Objetivo:** calcular cuánto gastó cada usuario en total (en pesos), considerando el precio de cada fruta y la cantidad comprada.

**Fórmula:** `gasto_linea = cantidad × precio_unitario` para cada fila de `compras.csv`.

**Pistas:**
1. Construye un diccionario de precios `{id_fruta: precio}` usando `reduce`.
2. Usa `map` para calcular el gasto de cada línea de compra: `(id_usuario, gasto_linea)`.
3. Usa `reduce` (`acumular`) para agrupar por usuario.

**Resultado esperado:** diccionario `{id_usuario: gasto_total}` en pesos.

In [3]:
# Tu solución aquí
gasto_por_usuario = {}  # {id_usuario: gasto_total}

for uid, gasto in list(gasto_por_usuario.items())[:5]:
    print(f"  Usuario {uid}: ${gasto}")

---

## Ejercicio 4 — El producto más vendido

**Objetivo:** encontrar cuál es la fruta que se vendió en mayor cantidad (suma de unidades).

**Resultado esperado:** nombre de la fruta y total de unidades vendidas.

**Pistas:**
1. Construye un diccionario de nombres de frutas `{id: nombre}` con `reduce`.
2. Usa `map` + `reduce` (`acumular`) para obtener la cantidad total por `id_producto`.
3. Usa `reduce` para encontrar el máximo del diccionario anterior.

In [ ]:
# Tu solución aquí
producto_mas_vendido = None
max_unidades = None

print(f"Producto más vendido: {producto_mas_vendido} con {max_unidades} unidades")

---

## Ejercicio 5 — Ingreso total por día

**Objetivo:** calcular el ingreso total (en pesos) generado en cada fecha.

**Resultado esperado:** diccionario `{fecha: ingreso_total}` ordenado cronológicamente.

**Pistas:**
- Usa el diccionario `precios` del ejercicio 3.
- Cada línea de compra tiene el campo `fecha_compra`; úsalo como clave del par.

In [5]:
# Tu solución aquí
ventas_por_dia = {}

for fecha, ingreso in list(ventas_por_dia.items())[:5]:
    print(f"  {fecha}: ${ingreso}")

---

## Ejercicio 6 — Top 3 usuarios por gasto

**Objetivo:** obtener los 3 usuarios que más gastaron, mostrando su **nombre** (no su id) y el gasto total.

**Pistas:**
1. Construye un diccionario `{id_usuario: nombre}` a partir de `usuarios` con `reduce`.
2. Usa `map` para convertir `gasto_por_usuario.items()` en pares `(nombre, gasto)`.
3. Usa `reduce` para mantener solo los 3 mayores.

> **Desafío extra:** ¿Puedes hacerlo usando *solo* `reduce`, sin `sorted`?

In [ ]:
# Tu solución aquí
top3 = None

print("Top 3 usuarios por gasto:")
for nombre, gasto in top3:
    print(f"  {nombre}: ${gasto}")

---

## Ejercicio 7 — Ticket promedio por usuario

**Objetivo:** calcular el **ticket promedio** de cada usuario:

$$\text{ticket promedio} = \frac{\text{gasto total del usuario}}{\text{número de órdenes distintas del usuario}}$$

Una **orden** es un `id_compra` único. Un mismo `id_compra` puede tener varias líneas de productos.

**Resultado esperado:** lista de tuplas `(nombre_usuario, ticket_promedio)` ordenada de mayor a menor, con el promedio redondeado a 2 decimales.

**Pistas:**
1. Con `reduce`, acumula por usuario un diccionario con dos campos: `gasto` (int) y `ordenes` (set de `id_compra`).
2. Con `map`, calcula el promedio: `gasto / len(ordenes)`.
3. Une con los nombres de usuarios usando `map` y el diccionario `nombre_usuario`.

In [ ]:
# Tu solución aquí
ticket_promedio = None  # lista de (nombre, promedio)

print("Ticket promedio por usuario:")
for nombre, ticket in ticket_promedio:
    print(f"  {nombre}: ${ticket}")

---

## Resumen

| Ejercicio | Concepto clave |
|-----------|---------------|
| 1 | `map` + `reduce` básico (suma global) |
| 2 | `map` → pares (clave, valor) + `reduce` para agrupar |
| 3 | Join entre colecciones + `reduce` anidado |
| 4 | `reduce` para encontrar el máximo |
| 5 | Agrupación por atributo fecha |
| 6 | `reduce` con acumulador de tamaño fijo (top-k) |
| 7 | Acumulador compuesto (gasto + conjunto de órdenes) |

### Patrón general

```python
from functools import reduce

# 1. MAP: producir pares (clave, valor)
pares = map(lambda x: (clave(x), valor(x)), datos)

# 2. REDUCE: combinar por clave
def acumular(acc, par):
    clave, valor = par
    acc[clave] = acc.get(clave, 0) + valor
    return acc

resultado = reduce(acumular, pares, {})  # {} es el acumulador inicial
```